# Práctica 08 — Modelos de lenguaje n-grama

**Curso:** NLP Clásico con Python · **Clase 08** · **Tiempo estimado:** 60-75 minutos
**Notebook de clase:** `notebooks_clase/clase08_modelos_lenguaje_ngramas.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los ejercicios 1 a 5 usan un corpus de cuatro frases y cifras que puedes calcular a mano. El reto usa el extracto del Quijote de `datos/gutenberg/` (en Colab, sube la carpeta `datos/`): un texto literario antiguo, no el español de hoy (*ejemplo ilustrativo*).
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Librerías: solo `numpy` y la biblioteca estándar.

---
## Celda 0 — Setup y datos

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("matplotlib", "matplotlib")
print("Librerías listas.")

In [ ]:
import os
import re
import math
import random
from collections import Counter

import numpy as np

random.seed(42)                          # semilla fija
np.random.seed(42)

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

INI, FIN, UNK = "<s>", "</s>", "<unk>"     # inicio de frase, fin de frase, fuera de vocabulario
ENCAB = re.compile(r"^(Capítulo|CAPITOLO)\b")   # dato-es

def capitulos(ruta):
    """Lee un extracto y devuelve una lista de capítulos (listas de párrafos)."""
    caps = []
    for parrafo in open(ruta, encoding="utf-8").read().split("\n\n"):
        parrafo = parrafo.strip()
        if not parrafo:
            continue
        if ENCAB.match(parrafo):
            caps.append([])
        else:
            caps[-1].append(parrafo)
    return caps

def oraciones(parrafos):
    """Divide en frases (. ! ? …) y en tokens en minúscula (sin puntuación)."""
    salida = []
    for p in parrafos:
        for o in re.split(r"(?<=[.!?…])\s+", p):
            toks = re.findall(r"\w+'|\w+", o.lower())
            if toks:
                salida.append(toks)
    return salida

print("Funciones auxiliares cargadas.")

In [ ]:
# datos-es
# Corpus minúsculo (4 frases por idioma) para calcular a mano
CHICO = {
    "spa": [["un", "gato", "come", "pescado"], ["un", "gato", "duerme"],
            ["un", "perro", "come", "carne"], ["un", "perro", "duerme"]],
    "ita": [["il", "gatto", "mangia", "pesce"], ["il", "gatto", "dorme"],
            ["il", "cane", "mangia", "carne"], ["il", "cane", "dorme"]],
}

# Conteos de bigramas del corpus minúsculo escritos a mano (español), para comprobar sin depender de otros ejercicios
BI_EJ = {("<s>", "un"): 4, ("un", "gato"): 2, ("un", "perro"): 2, ("gato", "come"): 1, ("gato", "duerme"): 1,
         ("perro", "come"): 1, ("perro", "duerme"): 1, ("come", "pescado"): 1, ("come", "carne"): 1,
         ("duerme", "</s>"): 2, ("pescado", "</s>"): 1, ("carne", "</s>"): 1}
CTX_EJ = {"<s>": 4, "un": 4, "gato": 2, "perro": 2, "come": 2, "duerme": 2, "pescado": 1, "carne": 1}
V_EJ = ["un", "gato", "perro", "come", "duerme", "pescado", "carne", FIN, UNK]   # |V| = 9

## Ejercicio 1 — Contar bigramas con `<s>` y `</s>`

Completa `contar_bigramas(oras)`. `oras` es una lista de frases (listas de tokens). Cada frase se rodea con `<s>` al inicio y `</s>` al final, y la función devuelve dos `Counter`: los bigramas `(v, w)` y los contextos `v` (cuántas veces `v` aparece como palabra anterior).

Antes de programar: en `CHICO["spa"]`, ¿cuántos bigramas distintos hay y cuánto vale el contexto `un`? (12 y 4.)

In [ ]:
def contar_bigramas(oras):
    bi, ctx = Counter(), Counter()
    for o in oras:
        seq = [INI] + o + [FIN]
        for v, w in zip(seq, seq[1:]):
            pass    # TODO: suma 1 a bi[(v, w)] y a ctx[v]
    return bi, ctx

# Comprobación automática
_bi, _ctx = contar_bigramas(CHICO["spa"])
assert isinstance(_bi, Counter) and len(_bi) > 0, "Completa las líneas marcadas con TODO"
assert len(_bi) == 12 and _ctx["un"] == 4 and _bi[("un", "gato")] == 2, "revisa los conteos: hay 12 bigramas distintos y c(un) = 4"
assert _bi[(INI, "un")] == 4 and _bi[("duerme", FIN)] == 2 and _ctx[INI] == 4, "no olvides <s> y </s>"
assert dict(_bi) == BI_EJ and dict(_ctx) == CTX_EJ
_bi2, _ = contar_bigramas(CHICO["ita"])
assert len(_bi2) == 12 and _bi2[("il", "gatto")] == 2
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Probabilidad del bigrama: MLE y Laplace

Completa `p_bigrama(w, v, bi, ctx, V, a=0.0)`: la probabilidad `P(w | v)` con `(c(v, w) + a) / (c(v) + a·|V|)`. Con `a = 0` es el estimador por conteo (MLE) y con `a = 1` es el suavizado de Laplace. Cuidado con el contexto `v` que no existe en `ctx`: con `a = 0` la probabilidad es `0.0` (no hay división por cero).

Calcula antes a mano con `BI_EJ` y `|V| = 9`: `P(gato | un)` = 1/2; con Laplace, `P(pescado | un)` = 1/13 y `P(gato | un)` = 3/13.

In [ ]:
def p_bigrama(w, v, bi, ctx, V, a=0.0):
    numerador = None    # TODO: c(v, w) + a
    denominador = None  # TODO: c(v) + a·|V|
    return None         # TODO: numerador / denominador, o 0.0 si el denominador es 0

# Comprobación automática
assert p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ) is not None, "Completa las líneas marcadas con TODO"
assert p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ) == 0.5 and p_bigrama("pescado", "un", BI_EJ, CTX_EJ, V_EJ) == 0.0
assert np.isclose(p_bigrama("pescado", "un", BI_EJ, CTX_EJ, V_EJ, 1.0), 1 / 13) and np.isclose(p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ, 1.0), 3 / 13), "con Laplace: (0 + 1) / (4 + 9) y (2 + 1) / (4 + 9)"
assert p_bigrama("gato", "zzz", BI_EJ, CTX_EJ, V_EJ) == 0.0, "contexto desconocido con a = 0: 0.0"
for _v in ("un", "come", "zzz"):
    for _a in (0.5, 1.0):
        assert np.isclose(sum(p_bigrama(w, _v, BI_EJ, CTX_EJ, V_EJ, _a) for w in V_EJ), 1.0), "con a > 0 las probabilidades condicionadas suman 1"
assert np.isclose(sum(p_bigrama(w, "un", BI_EJ, CTX_EJ, V_EJ) for w in V_EJ), 1.0)
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Perplejidad de un modelo de bigramas

Completa `perplejidad_bigrama(oras, prob)`. `prob(w, v)` devuelve la probabilidad de `w` dado `v`. La perplejidad es `exp(-(1/N) · Σ ln P)`, donde la suma recorre cada token y el `</s>` final de cada frase (con `<s>` como contexto inicial) y `N` cuenta esos tokens, **sin** contar `<s>`. Si alguna probabilidad es 0, devuelve `float("inf")`.

A mano: con MLE sobre `CHICO`, la frase `un perro duerme` tiene `P = 1/4` y `N = 4`: perplejidad `√2 ≈ 1,414`. Un modelo uniforme sobre 9 palabras tiene perplejidad 9.

In [ ]:
def perplejidad_bigrama(oras, prob):
    logsum, N = 0.0, 0
    for o in oras:
        seq = [INI] + o + [FIN]
        for v, w in zip(seq, seq[1:]):
            p = prob(w, v)
            # TODO: si p <= 0 devuelve float("inf"); si no, suma math.log(p) a logsum
            N += 1
    return None      # TODO: math.exp(-logsum / N)

# Comprobación automática
_mle = lambda w, v: BI_EJ.get((v, w), 0) / CTX_EJ[v] if v in CTX_EJ else 0.0
_pp = perplejidad_bigrama([["un", "perro", "duerme"]], _mle)
assert _pp is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_pp, math.sqrt(2)), "P = 1 · 1/2 · 1/2 · 1 = 1/4 con N = 4 tokens (contando </s>): perplejidad √2"
assert np.isclose(perplejidad_bigrama([["un", "perro", "duerme"], ["un", "gato", "come", "carne"]], lambda w, v: 1 / 9), 9.0), "un modelo uniforme sobre 9 palabras tiene perplejidad 9"
assert perplejidad_bigrama([["un", "pescado"]], _mle) == float("inf"), "un bigrama no visto (probabilidad 0) hace la perplejidad infinita"
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Vocabulario y tratamiento del OOV

Completa dos funciones: `construir_vocab(oras, minimo)` devuelve el conjunto de palabras con al menos `minimo` apariciones **en las frases que se le pasan** (el entrenamiento), y `reemplazar_oov(oras, vocab)` devuelve las frases con `<unk>` en lugar de cada palabra que no esté en `vocab`. Regla: el vocabulario se construye **solo con el entrenamiento** y se aplica igual a la prueba.

Cuenta a mano con `[["a","b","a"],["a","c"],["b","d","a"]]`: `a` aparece 4 veces, `b` 2, `c` y `d` 1. Con `minimo = 2` el vocabulario es `{a, b}`.

In [ ]:
def construir_vocab(oras, minimo):
    cuenta = Counter(t for o in oras for t in o)
    return None      # TODO: conjunto de las palabras con cuenta[t] >= minimo

def reemplazar_oov(oras, vocab):
    return None      # TODO: lista de frases donde cada token fuera de vocab se cambia por UNK

# Comprobación automática
_oras = [["a", "b", "a"], ["a", "c"], ["b", "d", "a"]]
_voc = construir_vocab(_oras, 2)
assert _voc is not None, "Completa las líneas marcadas con TODO"
assert _voc == {"a", "b"} and construir_vocab(_oras, 1) == {"a", "b", "c", "d"} and construir_vocab(_oras, 5) == set()
assert reemplazar_oov(_oras, _voc) == [["a", "b", "a"], ["a", UNK], ["b", UNK, "a"]], "cada palabra fuera del vocabulario pasa a <unk>"
assert reemplazar_oov([["a", "z"]], _voc) == [["a", UNK]] and _oras[1] == ["a", "c"], "no modifiques la lista original"
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Dividir por capítulos contiguos

Completa `dividir_por_capitulos(caps, n_train, n_dev)`. `caps` es una lista de capítulos (cada uno una lista de párrafos). Devuelve `(train, dev, test)`, cada uno como lista de frases (usa `oraciones`): `train` son los primeros `n_train` capítulos, `dev` los `n_dev` siguientes y `test` **todos los restantes**. Sin mezclar ni barajar: un modelo real se usa sobre texto que llega después.

In [ ]:
def dividir_por_capitulos(caps, n_train, n_dev):
    # TODO: aplana con oraciones(...) los capítulos caps[:n_train], luego los n_dev siguientes y luego el resto
    train = dev = test = None
    return train, dev, test

# Comprobación automática
_caps = [["a a. b b."], ["c c."], ["d d. e e.", "f f."], ["g g."], ["h h. i i."]]
_tr, _dv, _te = dividir_por_capitulos(_caps, 3, 1)
assert _tr is not None, "Completa las líneas marcadas con TODO"
assert _tr == [["a", "a"], ["b", "b"], ["c", "c"], ["d", "d"], ["e", "e"], ["f", "f"]], "train: los tres primeros capítulos, en orden"
assert _dv == [["g", "g"]] and _te == [["h", "h"], ["i", "i"]], "dev: el capítulo siguiente; test: el resto"
_, _, _te2 = dividir_por_capitulos(_caps, 2, 2)
assert _te2 == [["h", "h"], ["i", "i"]]
print("Ejercicio 5 correcto ✔")

---
## Reto final — Interpolar bigrama y unigrama sobre el Quijote

Con tus funciones, entrena sobre `datos/gutenberg/quijote_cap01-14.txt` (o el italiano, cambiando la ruta): 10 capítulos de entrenamiento, 2 de desarrollo y 2 de prueba, `minimo = 2` para el vocabulario. Escribe `p_interp(w, v, lam)`: `lam · P_bigrama(w | v) + (1 − lam) · P_unigrama(w)` (usa el bigrama MLE con `a = 0`; si el contexto no existe, usa solo el unigrama). Elige `lam` entre `LAMBDAS` con la perplejidad de **desarrollo** y calcula la perplejidad de **prueba**. Debe ser finita y mejor que la de Laplace `a = 1` del bigrama. Recuerda: es un corpus literario de hace cuatro siglos, no el español de hoy.

In [ ]:
LAMBDAS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
_ruta, _ntr, _ndv = "gutenberg/quijote_cap01-14.txt", 10, 2

caps = capitulos(cargar_datos(_ruta))
train, dev, test = dividir_por_capitulos(caps, _ntr, _ndv)
vocab = construir_vocab(train, 2)
train_u, dev_u, test_u = (reemplazar_oov(x, vocab) for x in (train, dev, test))
bi, ctx = contar_bigramas(train_u)
V = sorted(vocab | {FIN, UNK})
uni = Counter(t for o in train_u for t in o + [FIN])
total = sum(uni.values())

def p_interp(w, v, lam):
    p_uni = uni[w] / total
    if ctx.get(v, 0) == 0:
        return None       # TODO: contexto desconocido: solo el unigrama
    return None           # TODO: lam · P_bigrama (MLE) + (1 − lam) · P_unigrama

# TODO: elige lam_mejor entre LAMBDAS con la perplejidad de desarrollo (dev_u) y calcula pp_test con test_u
lam_mejor, pp_test = None, None
pp_laplace = perplejidad_bigrama(test_u, lambda w, v: p_bigrama(w, v, bi, ctx, V, 1.0))

# Comprobación automática
assert lam_mejor in LAMBDAS and pp_test is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(sum(p_interp(w, "un" if "un" in ctx else next(iter(ctx)), lam_mejor) for w in V), 1.0), "la interpolación debe sumar 1 sobre V"
assert np.isclose(sum(p_interp(w, "zzz", lam_mejor) for w in V), 1.0), "y también con un contexto desconocido"
assert all(perplejidad_bigrama(dev_u, lambda w, v: p_interp(w, v, lam_mejor)) <= perplejidad_bigrama(dev_u, lambda w, v, l=l: p_interp(w, v, l)) + 1e-9 for l in LAMBDAS), "lam_mejor debe minimizar la perplejidad de desarrollo"
assert np.isclose(pp_test, perplejidad_bigrama(test_u, lambda w, v: p_interp(w, v, lam_mejor)))
assert math.isfinite(pp_test) and pp_test < pp_laplace, "debe ser finita y mejor que Laplace a = 1"
print("Reto final correcto ✔")
print("lam =", lam_mejor, "| PP test =", round(pp_test, 1), "| PP Laplace =", round(pp_laplace, 1))

**Tus conclusiones (escribe 3-4 frases):** ¿por qué el suavizado de Laplace con `a = 1` empeora tanto un modelo con vocabulario grande y pocos datos? ¿Por qué no se pueden comparar directamente la perplejidad de un modelo con `minimo = 2` y la de otro con `minimo = 20`, ni la de un texto en español con la de uno en italiano?

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def contar_bigramas(oras):
    bi, ctx = Counter(), Counter()
    for o in oras:
        seq = [INI] + o + [FIN]
        for v, w in zip(seq, seq[1:]):
            bi[(v, w)] += 1
            ctx[v] += 1
    return bi, ctx

# Comprobación automática
_bi, _ctx = contar_bigramas(CHICO["spa"])
assert isinstance(_bi, Counter) and len(_bi) > 0, "Completa las líneas marcadas con TODO"
assert len(_bi) == 12 and _ctx["un"] == 4 and _bi[("un", "gato")] == 2, "revisa los conteos: hay 12 bigramas distintos y c(un) = 4"
assert _bi[(INI, "un")] == 4 and _bi[("duerme", FIN)] == 2 and _ctx[INI] == 4, "no olvides <s> y </s>"
assert dict(_bi) == BI_EJ and dict(_ctx) == CTX_EJ
_bi2, _ = contar_bigramas(CHICO["ita"])
assert len(_bi2) == 12 and _bi2[("il", "gatto")] == 2
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
def p_bigrama(w, v, bi, ctx, V, a=0.0):
    numerador = bi.get((v, w), 0) + a
    denominador = ctx.get(v, 0) + a * len(V)
    return numerador / denominador if denominador else 0.0

# Comprobación automática
assert p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ) is not None, "Completa las líneas marcadas con TODO"
assert p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ) == 0.5 and p_bigrama("pescado", "un", BI_EJ, CTX_EJ, V_EJ) == 0.0
assert np.isclose(p_bigrama("pescado", "un", BI_EJ, CTX_EJ, V_EJ, 1.0), 1 / 13) and np.isclose(p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ, 1.0), 3 / 13), "con Laplace: (0 + 1) / (4 + 9) y (2 + 1) / (4 + 9)"
assert p_bigrama("gato", "zzz", BI_EJ, CTX_EJ, V_EJ) == 0.0, "contexto desconocido con a = 0: 0.0"
for _v in ("un", "come", "zzz"):
    for _a in (0.5, 1.0):
        assert np.isclose(sum(p_bigrama(w, _v, BI_EJ, CTX_EJ, V_EJ, _a) for w in V_EJ), 1.0), "con a > 0 las probabilidades condicionadas suman 1"
assert np.isclose(sum(p_bigrama(w, "un", BI_EJ, CTX_EJ, V_EJ) for w in V_EJ), 1.0)
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
def perplejidad_bigrama(oras, prob):
    logsum, N = 0.0, 0
    for o in oras:
        seq = [INI] + o + [FIN]
        for v, w in zip(seq, seq[1:]):
            p = prob(w, v)
            if p <= 0:
                return float("inf")
            logsum += math.log(p)
            N += 1
    return math.exp(-logsum / N)

# Comprobación automática
_mle = lambda w, v: BI_EJ.get((v, w), 0) / CTX_EJ[v] if v in CTX_EJ else 0.0
_pp = perplejidad_bigrama([["un", "perro", "duerme"]], _mle)
assert _pp is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_pp, math.sqrt(2)), "P = 1 · 1/2 · 1/2 · 1 = 1/4 con N = 4 tokens (contando </s>): perplejidad √2"
assert np.isclose(perplejidad_bigrama([["un", "perro", "duerme"], ["un", "gato", "come", "carne"]], lambda w, v: 1 / 9), 9.0), "un modelo uniforme sobre 9 palabras tiene perplejidad 9"
assert perplejidad_bigrama([["un", "pescado"]], _mle) == float("inf"), "un bigrama no visto (probabilidad 0) hace la perplejidad infinita"
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
def construir_vocab(oras, minimo):
    cuenta = Counter(t for o in oras for t in o)
    return {t for t, k in cuenta.items() if k >= minimo}

def reemplazar_oov(oras, vocab):
    return [[t if t in vocab else UNK for t in o] for o in oras]

# Comprobación automática
_oras = [["a", "b", "a"], ["a", "c"], ["b", "d", "a"]]
_voc = construir_vocab(_oras, 2)
assert _voc is not None, "Completa las líneas marcadas con TODO"
assert _voc == {"a", "b"} and construir_vocab(_oras, 1) == {"a", "b", "c", "d"} and construir_vocab(_oras, 5) == set()
assert reemplazar_oov(_oras, _voc) == [["a", "b", "a"], ["a", UNK], ["b", UNK, "a"]], "cada palabra fuera del vocabulario pasa a <unk>"
assert reemplazar_oov([["a", "z"]], _voc) == [["a", UNK]] and _oras[1] == ["a", "c"], "no modifiques la lista original"
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
def dividir_por_capitulos(caps, n_train, n_dev):
    def aplanar(cs):
        return [o for cap in cs for o in oraciones(cap)]
    return aplanar(caps[:n_train]), aplanar(caps[n_train:n_train + n_dev]), aplanar(caps[n_train + n_dev:])

# Comprobación automática
_caps = [["a a. b b."], ["c c."], ["d d. e e.", "f f."], ["g g."], ["h h. i i."]]
_tr, _dv, _te = dividir_por_capitulos(_caps, 3, 1)
assert _tr is not None, "Completa las líneas marcadas con TODO"
assert _tr == [["a", "a"], ["b", "b"], ["c", "c"], ["d", "d"], ["e", "e"], ["f", "f"]], "train: los tres primeros capítulos, en orden"
assert _dv == [["g", "g"]] and _te == [["h", "h"], ["i", "i"]], "dev: el capítulo siguiente; test: el resto"
_, _, _te2 = dividir_por_capitulos(_caps, 2, 2)
assert _te2 == [["h", "h"], ["i", "i"]]
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
LAMBDAS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
_ruta, _ntr, _ndv = "gutenberg/quijote_cap01-14.txt", 10, 2

caps = capitulos(cargar_datos(_ruta))
train, dev, test = dividir_por_capitulos(caps, _ntr, _ndv)
vocab = construir_vocab(train, 2)
train_u, dev_u, test_u = (reemplazar_oov(x, vocab) for x in (train, dev, test))
bi, ctx = contar_bigramas(train_u)
V = sorted(vocab | {FIN, UNK})
uni = Counter(t for o in train_u for t in o + [FIN])
total = sum(uni.values())

def p_interp(w, v, lam):
    p_uni = uni[w] / total
    if ctx.get(v, 0) == 0:
        return p_uni
    return lam * bi.get((v, w), 0) / ctx[v] + (1 - lam) * p_uni

lam_mejor = min(LAMBDAS, key=lambda l: perplejidad_bigrama(dev_u, lambda w, v: p_interp(w, v, l)))
pp_test = perplejidad_bigrama(test_u, lambda w, v: p_interp(w, v, lam_mejor))
pp_laplace = perplejidad_bigrama(test_u, lambda w, v: p_bigrama(w, v, bi, ctx, V, 1.0))

assert lam_mejor in LAMBDAS and pp_test is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(sum(p_interp(w, "un" if "un" in ctx else next(iter(ctx)), lam_mejor) for w in V), 1.0), "la interpolación debe sumar 1 sobre V"
assert np.isclose(sum(p_interp(w, "zzz", lam_mejor) for w in V), 1.0), "y también con un contexto desconocido"
assert all(perplejidad_bigrama(dev_u, lambda w, v: p_interp(w, v, lam_mejor)) <= perplejidad_bigrama(dev_u, lambda w, v, l=l: p_interp(w, v, l)) + 1e-9 for l in LAMBDAS), "lam_mejor debe minimizar la perplejidad de desarrollo"
assert np.isclose(pp_test, perplejidad_bigrama(test_u, lambda w, v: p_interp(w, v, lam_mejor)))
assert math.isfinite(pp_test) and pp_test < pp_laplace, "debe ser finita y mejor que Laplace a = 1"
print("Reto final correcto ✔")
print("lam =", lam_mejor, "| PP test =", round(pp_test, 1), "| PP Laplace =", round(pp_laplace, 1))

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto no te ha quedado claro (regla de la cadena frente a hipótesis de Markov, `<s>` y `</s>`, suavizado, interpolación, perplejidad, OOV)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*